# BAB VII. Visualisasi dan Analisis Hubungan Antarvariabel (Data Visualization and Relationship Analysis)

Visualisasi membantu mengamati distribusi dan asosiasi, bukan membuktikan sebab-akibat. Pilih variabel berdasarkan kamus data dan hasil inspeksi.

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import PROCESSED_DATA_DIR
from eda_knkt.ingestion import load_csv
from eda_knkt.cleaning import replace_placeholders
from eda_knkt.transformation import convert_numeric_columns, add_distance_to_stop_feature

DATA_FILE = PROCESSED_DATA_DIR / 'data_logger_knkt_tabel_6_10.csv'
df_raw = load_csv(DATA_FILE)
df_missing = replace_placeholders(df_raw, ['---'])
NUMERIC_OPERATIONAL_COLUMNS = [
    'km', 'm', 'Speed (km/h)', 'Acceleration (m/sec²)',
    'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)'
]
df_transformed = convert_numeric_columns(
    df_missing, NUMERIC_OPERATIONAL_COLUMNS, errors='coerce'
)
df_transformed = add_distance_to_stop_feature(df_transformed)
df_transformed['Timestamp'] = pd.to_datetime(
    df_transformed['Date'].astype(str) + ' ' + df_transformed['Time'].astype(str),
    format='%d/%m/%Y %H.%M.%S', errors='raise'
)
df_transformed = df_transformed.sort_values('Timestamp').reset_index(drop=True)
df_transformed['Elapsed (s)'] = (
    df_transformed['Timestamp'] - df_transformed['Timestamp'].iloc[0]
).dt.total_seconds()
print(f'Loaded {len(df_raw)} records × {len(df_raw.columns)} columns from {DATA_FILE.name}')

In [ ]:
import matplotlib.pyplot as plt
from eda_knkt.visualization import (
    plot_numeric_distribution,
    plot_categorical_counts,
    plot_scatter,
    plot_correlation_heatmap,
)
from eda_knkt.statistics import correlation_matrix

if "df_transformed" not in globals():
    print("Jalankan notebook 06 terlebih dahulu.")
else:
    numeric_columns = [c for c in df_transformed.select_dtypes(include="number").columns if c not in {"km", "m"}]
    categorical_columns = ["Throttle", "PCS Open", "IBS", "DB Excit", "Dir Call", "DB Set", "DB Star"]
    print("Numeric:", numeric_columns)
    print("Categorical:", categorical_columns)

    # Contoh: pilih satu kolom setelah makna dan satuannya diverifikasi.
    if numeric_columns:
        ax = plot_numeric_distribution(df_transformed, numeric_columns[0], bins=20)
        plt.show()
    if categorical_columns:
        ax = plot_categorical_counts(df_transformed, categorical_columns[0])
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

In [ ]:
# Korelasi hanya dihitung pada variabel numerik terpilih yang bermakna.
# Pilih kolom secara eksplisit setelah memeriksa kamus data.
CORRELATION_COLUMNS = ['distance_to_stop_m', 'Speed (km/h)', 'Acceleration (m/sec²)', 'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)']

if CORRELATION_COLUMNS:
    corr = correlation_matrix(df_transformed, CORRELATION_COLUMNS, method="spearman")
    display(corr)
    ax = plot_correlation_heatmap(corr)
    plt.tight_layout()
    plt.show()
else:
    print("Tentukan CORRELATION_COLUMNS setelah verifikasi variabel.")

## Interpretasi visual
- Histogram/box plot: distribusi dan rentang variabel numerik.
- Bar chart: frekuensi kategori.
- Scatter plot: pola pasangan variabel numerik.
- Heatmap: kekuatan dan arah asosiasi sesuai metode korelasi.

Pearson mengukur asosiasi linear; Spearman dan Kendall berbasis peringkat. Korelasi tidak membuktikan pengaruh kausal. Hindari istilah “memengaruhi” jika desain analisis tidak mendukung inferensi kausal.